In [107]:
from sqlite3 import OperationalError
from datetime import datetime, timedelta, timezone

import numpy as np
import pandas as pd
import pandas_market_calendars as mcal
import quantstats as qs

from ibkr_data.db import table_name, get_connection

conn = get_connection()
cal = mcal.get_calendar("NYSE")

In [108]:
start = (datetime.now(timezone.utc) - timedelta(days=365*5)).date()
end = datetime.now(timezone.utc).date()

schedule = cal.schedule(start_date=start, end_date=end)
market_days = [d.date() for d in schedule.index]
print(f"{len(market_days)} market days from {start} to {end}")

1255 market days from 2021-06-08 to 2026-06-07


In [109]:
tickers = conn.execute(
    """
    SELECT REPLACE(REPLACE(name, 'ibkr_', ''), '_5m', '') AS wildcard_value
    FROM sqlite_schema 
    WHERE type='table' AND name LIKE 'ibkr_%_5m' order by random() limit 150;
    """
).fetchall()
tickers = [r[0] for r in tickers]
print(f"{len(tickers)} tickers")

print(tickers)

150 tickers
['MSCI', 'HUM', 'LHX', 'CARR', 'CCI', 'KEY', 'KR', 'MCD', 'DHR', 'LUV', 'IBM', 'MCHP', 'CTVA', 'NVDA', 'DLR', 'ADP', 'DE', 'KEYS', 'AVB', 'UNH', 'ABT', 'NOW', 'LYV', 'FCX', 'MSFT', 'EOG', 'AMD', 'CME', 'CB', 'AZN', 'DGX', 'ASML', 'CAG', 'MOS', 'ARE', 'NKE', 'CMI', 'HD', 'CI', 'A', 'WFC', 'MDLZ', 'GE', 'EL', 'EEM', 'PG', 'MS', 'INTC', 'GIS', 'MRNA', 'AKAM', 'BDX', 'EA', 'GEHC', 'LYB', 'LIN', 'DLTR', 'NEM', 'AMT', 'ENPH', 'JBHT', 'CVS', 'BKR', 'HPQ', 'DDOG', 'KMI', 'ARM', 'FSLR', 'KLAC', 'MMM', 'KHC', 'ECL', 'ADBE', 'EW', 'ILMN', 'V', 'DELL', 'CF', 'MET', 'EQR', 'BA', 'CDNS', 'ALB', 'CAT', 'NOC', 'CSCO', 'FTNT', 'COP', 'LLY', 'ITW', 'LOW', 'DAL', 'FANG', 'DG', 'JNJ', 'CRM', 'COST', 'DOW', 'WMT', 'BSX', 'EMN', 'CRWD', 'KO', 'LRCX', 'DIS', 'GOOGL', 'ABBV', 'HBAN', 'FFIV', 'FDX', 'DD', 'NSC', 'AZO', 'FITB', 'AVGO', 'IR', 'CHTR', 'FOXA', 'TSLA', 'CPB', 'AMGN', 'C', 'GILD', 'GS', 'AMZN', 'NEE', 'CVX', 'CSX', 'DVN', 'NFLX', 'AIG', 'MU', 'BIIB', 'EMR', 'LH', 'INTU', 'JPM', 'LMT', 'M

In [110]:
slippage = 0.0001        # 0.01% per trade (applied to entry)

# IBKR Pro Tiered fees (US stocks, ≤ 300K shares/month)
commission_per_share = 0.0035    # IBKR base commission
exchange_fee = 0.0030            # Exchange pass-through (taking liquidity, blended)
clearing_fee = 0.0002            # NSCC/DTC clearing
finra_cat = 0.000003             # FINRA CAT
sec_fee_rate = 0.0000206         # SEC fee on sells only
minimum_per_order = 0.35
maximum_per_order_pct = 0.01     # 1% of trade value

def trade_cost(shares, price, sell=False):
    """IBKR Pro Tiered all-in cost for one side of a US stock trade."""
    per_share = commission_per_share + exchange_fee + clearing_fee + finra_cat
    fee = max(shares * per_share, minimum_per_order)
    fee = min(fee, shares * price * maximum_per_order_pct)
    if sell:
        fee += shares * price * sec_fee_rate  # SEC fee on sells
    return fee

risk_per_trade = 0.01   # 1% of AUM risked per trade
best_n_stocks = 5
atr_distance = 0.1
relative_volume_threshold = 1.
equity = []
portfolio_value = 1_000

monthly_data_fee = 4.5
last_month = None

for day in market_days:
    daily_opening_range_volume = []
    for ticker in tickers:
        tbl = table_name(ticker, timeframe="5m_first")
        try:
            row = conn.execute(
                f"SELECT ts, relative_volume FROM [{tbl}] WHERE date(ts) = ? and relative_volume is not null",
                (day.isoformat(),),
            ).fetchone()
        except OperationalError:
            row = None

        if row is None:
            continue

        daily_opening_range_volume.append((ticker, row[1]))

    daily_opening_range_volume = sorted(
        daily_opening_range_volume, key=lambda x: x[1], reverse=True
    )

    stocks_in_play = daily_opening_range_volume[:best_n_stocks]

    trade_made_today = False
    for stock in stocks_in_play:
        if stock[1] < relative_volume_threshold:
            continue
        five_min_table = table_name(ticker=stock[0], timeframe="5m")
        daily_table = table_name(ticker=stock[0], timeframe="1d")

        five_min = conn.execute(
            f"select ts, open, high, low, close from [{five_min_table}] where date(ts) = ? order by ts asc",
            (day.isoformat(),),
        ).fetchall()

        daily = conn.execute(
            f"select atr, close from [{daily_table}] where date(ts) = ?",
            (day.isoformat(),),
        ).fetchone()

        if not five_min or not daily:
            continue

        first_low = five_min[0][3]
        first_high = five_min[0][2]
        first_open = five_min[0][1]
        first_close = five_min[0][4]

        delta = np.sign(first_close - first_open)

        direction = np.sign(delta)
        stop_loss = daily[0] * atr_distance
        if daily[0] is None or np.isnan(daily[0]):
            continue

        long_signal = direction == 1
        short_signal = direction == -1
        

        if short_signal:
            stop_order_entry = first_low
            stop_order_stop_loss = first_low + stop_loss
        elif long_signal:
            stop_order_entry = first_high
            stop_order_stop_loss = first_high - stop_loss
        else:
            continue

        is_trade_open = False
        entry_price = 0
        shares = 0

        for candle in five_min[1:-1]:
            
            low, high = candle[3], candle[2]
            if not is_trade_open:
                if short_signal and not trade_made_today and low <= stop_order_entry:
                    entry_price = stop_order_entry * (1 - slippage)
                    risk_distance = abs(stop_order_stop_loss - entry_price)
                    shares = int((portfolio_value * risk_per_trade) / risk_distance) if risk_distance > 0 and portfolio_value == portfolio_value else 0
                    if shares:
                        portfolio_value += shares * entry_price - trade_cost(shares, entry_price, sell=True)
                        is_trade_open = True
                        trade_made_today = True
                elif long_signal and not trade_made_today and high >= stop_order_entry:
                    entry_price = stop_order_entry * (1 + slippage)
                    risk_distance = abs(stop_order_stop_loss - entry_price)
                    shares = int((portfolio_value * risk_per_trade) / risk_distance) if risk_distance > 0 and portfolio_value == portfolio_value else 0
                    if shares:
                        cost = shares * entry_price + trade_cost(shares, entry_price, sell=False)
                        if cost > portfolio_value:
                            shares = 0
                            continue
                        portfolio_value -= cost
                        is_trade_open = True
                        trade_made_today = True
            else:
                if short_signal and high >= stop_order_stop_loss:
                    portfolio_value -= shares * stop_order_stop_loss + trade_cost(shares, stop_order_stop_loss, sell=False)
                    is_trade_open = False
                elif long_signal and low <= stop_order_stop_loss:
                    portfolio_value += shares * stop_order_stop_loss - trade_cost(shares, stop_order_stop_loss, sell=True)
                    is_trade_open = False
        # --- close any open position at last candle ---
        if is_trade_open:
            last_candle = five_min[-1]
            close_price = last_candle[4]
            if short_signal:
                portfolio_value -= shares * close_price + trade_cost(shares, close_price, sell=False)
            elif long_signal:
                portfolio_value += shares * close_price - trade_cost(shares, close_price, sell=True)
            is_trade_open = False
        
        

    # --- track daily portfolio value ---
    equity.append(
        {"date": day, "portfolio_value": portfolio_value}
    )


    if last_month is not None and day.month != last_month:
        portfolio_value -= monthly_data_fee
    last_month = day.month
    
print(f"{len(equity)} days processed")

1255 days processed


In [111]:
returns = (
    pd.DataFrame(equity)
    .set_index(pd.to_datetime(pd.DataFrame(equity)["date"]))["portfolio_value"]
    .pct_change()
    .dropna()
)

# from ibkr_data.db import get_connection, count_5m_tickers
# conn = get_connection()
# n_tickers = count_5m_tickers(conn)

qs.reports.html(returns, output="backtest.html", title=f"Strategy Backtest ({len(tickers)} tickers)")


findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font f